<a href="https://www.kaggle.com/code/adithyan65/malayalamcybercon-data?scriptVersionId=323008607" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# MalayalamCyberCon — GPU Training Notebook
### Conflict & Cyberbullying Detection in Manglish YouTube Comments

---

## Before Running

| Step | Action |
|------|--------|
| **1. Accelerator** | Settings → Accelerator → **GPU T4 x2** |
| **2. Dataset** | Add Data → your `malayalamcybercon-dataset` (train/val/test/clean CSVs) |
| **3. Run** | Run All — takes ~30–40 min |

---

## Label Schema (0-indexed in splits)

| Task | Labels | Notes |
|------|--------|-------|
| `label_conflict` | 0 = no conflict, 1 = conflict | Binary |
| `label_severity` | 0 = mild, 1 = moderate, 2 = severe | conflict=1 only |
| `label_type` | 0 = personal, 1 = political, 2 = sexual/gendered, 3 = threat | conflict=1 only, single-label |

---

## Model
**google/muril-base-cased** — pretrained on 17 Indian languages including transliterated (Roman-script) Malayalam. Chosen over XLM-RoBERTa for better Manglish coverage.

In [5]:
%%capture
!pip install transformers accelerate scikit-learn

## 1 — Install Dependencies

MODEL_NAME  = 'google/muril-base-cased'
DATA_DIR    = '/kaggle/input/datasets/adithyan65/malayalamcybercon-dataset'
OUTPUT_DIR  = '/kaggle/working/models'
BATCH_SIZE  = 16
LR          = 2e-5
MAX_LEN     = 256
SEED        = 42
TASKS       = ['conflict', 'severity', 'type']

# Severity and type get more epochs — smaller datasets need longer training
EPOCHS = {'conflict': 5, 'severity': 10, 'type': 10}

In [6]:
# ── Disk cleanup ──────────────────────────────────────────────────────────────
import shutil
from pathlib import Path
for d in Path('/kaggle/working').iterdir():
    try:
        shutil.rmtree(d) if d.is_dir() else d.unlink()
    except: pass
_, used, free = shutil.disk_usage('/kaggle/working')
print(f'Disk: {free/1e9:.1f} GB free')

Disk: 20.9 GB free


## 3 — Imports & Device Check

## 4 — Load Data Splits

## 5 — Model Components
### Dataset & Focal Loss Trainer

**Why focal loss?** Standard cross-entropy treats all examples equally. Focal loss down-weights easy (correctly classified) examples so the model focuses on hard cases — important for imbalanced classes like `threat` (2.7% of conflict rows).

**Why `load_best_model_at_end=True`?** HuggingFace Trainer saves a checkpoint after each epoch and automatically reloads the best-performing one at the end of training. Combined with `trainer.save_model()`, this guarantees the saved model is the best epoch, not the last.

def train_task(task, train_all, val_all, test_all, tokenizer):
    label_col  = f'label_{task}'
    num_labels = 2 if task == 'conflict' else (3 if task == 'severity' else 4)

    if task == 'conflict':
        train_rows, val_rows, test_rows = train_all, val_all, test_all
    else:
        train_rows = [r for r in train_all if r['label_conflict'] == 1]
        val_rows   = [r for r in val_all   if r['label_conflict'] == 1]
        test_rows  = [r for r in test_all  if r['label_conflict'] == 1]

    print(f'\n{"─"*60}')
    print(f'  Task: {task.upper()}  |  num_labels={num_labels}  |  '
          f'train:{len(train_rows)}  val:{len(val_rows)}  test:{len(test_rows)}')
    print(f'{"─"*60}')

    if len(train_rows) < 20:
        print('  [SKIP] Not enough samples.')
        return None

    arr     = np.array([r[label_col] for r in train_rows])
    classes = np.unique(arr)
    weights = compute_class_weight('balanced', classes=classes, y=arr)
    class_weights = torch.tensor(weights, dtype=torch.float)
    print('  Class weights: ' + '  '.join(f'class {c}->{w:.3f}' for c,w in zip(classes,weights)))

    gamma = 0.0 if task == 'conflict' else (1.0 if task == 'severity' else 2.0)
    print(f'  Focal loss gamma: {gamma}')

    train_ds = ThreadDataset(train_rows, tokenizer, MAX_LEN, label_col)
    val_ds   = ThreadDataset(val_rows,   tokenizer, MAX_LEN, label_col)
    test_ds  = ThreadDataset(test_rows,  tokenizer, MAX_LEN, label_col)

    task_out = Path(OUTPUT_DIR) / task
    best_dir = task_out / 'best'
    best_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

    training_args = TrainingArguments(
        output_dir                  = str(task_out),
        num_train_epochs            = EPOCHS[task],
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        learning_rate               = LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        save_total_limit            = 2,              # keep only last 2 checkpoints to save disk
        load_best_model_at_end      = True,           # Trainer loads best checkpoint after training
        metric_for_best_model       = 'f1_macro',
        greater_is_better           = True,
        logging_steps               = 20,
        seed                        = SEED,
        report_to                   = 'none',
        fp16                        = torch.cuda.is_available(),
    )

    trainer = FocalLossTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = make_compute_metrics(task),
        class_weights   = class_weights,
        gamma           = gamma,
    )

    print(f'  Training on {device}...')
    trainer.train()
    # Trainer automatically loads best checkpoint at end (load_best_model_at_end=True)

    # Test evaluation on best model
    preds_out = trainer.predict(test_ds)
    y_pred = np.argmax(preds_out.predictions, axis=-1).tolist()
    y_true = [r[label_col] for r in test_rows]
    print(f'\n  Test set report ({task}):')
    print_report(task, y_true, y_pred)

    # trainer.save_model guarantees saving the best-epoch model (not last epoch)
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    print(f'  Saved -> {best_dir}')
    return best_dir

## 7 — Training Function

Three separate classifiers, one per task:
- **Conflict** — trained on all rows, gamma=0 (standard CE), 5 epochs
- **Severity** — trained on conflict=1 rows only, gamma=1.0, 10 epochs  
- **Type** — trained on conflict=1 rows only, gamma=2.0 (strongest focus on rare classes), 10 epochs

## 8 — Run Training

---
## After Downloading

1. Extract `malayalamcybercon_models.zip` into your local `models/` folder
2. Verify models are trained (not near-random):
```python
from safetensors import safe_open
for task in ['conflict', 'severity', 'type']:
    with safe_open(f'models/{task}/best/model.safetensors', framework='pt', device='cpu') as f:
        w = f.get_tensor('classifier.weight')
        print(f'{task}: mean_abs={w.abs().mean():.4f}')
```
3. Run inference:
```bash
python src/predict.py --text "[1] poda thayoli [2★] ninte ammayude poor"
```

### Expected Results
| Task | Metric | Target |
|------|--------|--------|
| Conflict | macro F1 | ≥ 0.77 |
| Severity | macro F1 | ≥ 0.40 (improvement over 0.24) |
| Type | macro F1 | > 0.30 (first real run) |

## 9 — Zip & Download Models

In [7]:
DATA_DIR    = '/kaggle/input/datasets/adithyan65/malayalamcybercon-dataset1'
OUTPUT_DIR  = '/kaggle/working/models'
MAX_LEN     = 256
SEED        = 42
TASKS       = ['conflict', 'severity', 'type']

LR_BY_TASK         = {'conflict': 2e-5,  'severity': 1e-5,  'type': 1e-5}
EPOCHS             = {'conflict': 5,     'severity': 10,    'type': 15}
BATCH_SIZE_DEFAULT = 16

# (HuggingFace model ID, short slug used for output dirs)
MODELS_TO_COMPARE = [
    ('xlm-roberta-base',        'xlmr_base'),
    ('google/muril-base-cased', 'muril_base'),
    ('xlm-roberta-large',       'xlmr_large'),
]

# xlm-roberta-large needs a smaller per-device batch to stay within T4 VRAM
BATCH_SIZE_OVERRIDE = {
    'xlmr_large': 8,
}

In [8]:
import copy, csv
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, TrainerCallback, set_seed,
)

set_seed(SEED)
device = 'GPU' if torch.cuda.is_available() else 'CPU'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

Device: GPU
GPU   : Tesla T4


In [9]:
# ── Load pre-split CSV files ───────────────────────────────────────────────────
def load_split(path):
    rows = []
    skipped = 0
    with open(path, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            lc = row.get('label_conflict', '').strip()
            ls = row.get('label_severity', '').strip()
            lt = row.get('label_type', '').strip()
            if not lc:
                continue
            lc_int = int(lc)
            if lc_int == 1 and not ls:
                skipped += 1
                continue
            ls_int = int(ls) if ls else 0
            lt_int = int(lt) if lt else 0
            rows.append({
                'text':           row['thread_text'],
                'label_conflict': lc_int,
                'label_severity': ls_int,
                'label_type':     lt_int,
            })
    if skipped:
        print(f'  [WARN] {path}: skipped {skipped} rows')
    return rows

train_all = load_split(f'{DATA_DIR}/train.csv')
val_all   = load_split(f'{DATA_DIR}/val.csv')
test_all  = load_split(f'{DATA_DIR}/test.csv')

for name, split in [('train', train_all), ('val', val_all), ('test', test_all)]:
    c1 = sum(1 for r in split if r['label_conflict'] == 1)
    print(f'{name:<6}: {len(split)} rows  conflict=1:{c1}({100*c1/len(split):.0f}%)')

train : 1168 rows  conflict=1:389(33%)
val   : 249 rows  conflict=1:83(33%)
test  : 252 rows  conflict=1:84(33%)


In [10]:
import random as _random

def oversample_to_balance(rows, label_col, seed=42):
    """Duplicate minority-class rows so every class matches the majority count."""
    rng = _random.Random(seed)
    counts = Counter(r[label_col] for r in rows)
    max_count = max(counts.values())
    balanced = list(rows)
    for cls, cnt in counts.items():
        deficit = max_count - cnt
        if deficit > 0:
            cls_rows = [r for r in rows if r[label_col] == cls]
            balanced.extend(rng.choices(cls_rows, k=deficit))
    rng.shuffle(balanced)
    return balanced


# ── Dataset ───────────────────────────────────────────────────────────────────
class ThreadDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, label_col):
        self.labels    = [r[label_col] for r in rows]
        self.encodings = tokenizer(
            [r['text'] for r in rows],
            truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# ── Focal loss trainer ────────────────────────────────────────────────────────
class FocalLossTrainer(Trainer):
    """
    - conflict/type : focal loss (gamma=0 / 2.0) with class weights
    - severity      : ordinal soft-label loss — transfers 5% probability mass to
                      each adjacent class so mild→severe is penalised more than
                      mild→moderate. Combined with class weights.
    """
    def __init__(self, *args, class_weights=None, gamma=2.0, task='conflict', **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma
        self.task  = task

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None

        if self.task == 'severity':
            # Build ordinal soft targets: 5% mass to each neighbour
            num_classes = logits.size(-1)
            soft = F.one_hot(labels, num_classes).float()
            transfer = 0.05
            for i in range(len(labels)):
                lbl = labels[i].item()
                if lbl > 0:
                    soft[i, lbl]     -= transfer
                    soft[i, lbl - 1] += transfer
                if lbl < num_classes - 1:
                    soft[i, lbl]     -= transfer
                    soft[i, lbl + 1] += transfer
            log_probs  = F.log_softmax(logits, dim=-1)
            loss_per_s = -(soft * log_probs).sum(dim=-1)
            if w is not None:
                loss_per_s = loss_per_s * w[labels]
            loss = loss_per_s.mean()
        else:
            ce   = F.cross_entropy(logits, labels, weight=w, reduction='none')
            pt   = torch.exp(-ce)
            loss = ((1 - pt) ** self.gamma * ce).mean()

        return (loss, outputs) if return_outputs else loss


# ── Best-model-in-memory callback ─────────────────────────────────────────────
class BestModelInMemory(TrainerCallback):
    def __init__(self):
        self.best_f1    = -1.0
        self.best_state = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        model = kwargs.get('model')
        if model is None:
            return
        f1 = (metrics or {}).get('eval_f1_macro', -1.0)
        if f1 > self.best_f1:
            self.best_f1    = f1
            self.best_state = {k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()}
            print(f'  * New best f1_macro: {f1:.4f} — saved in memory')

In [11]:
# ── Metrics ───────────────────────────────────────────────────────────────────
def make_compute_metrics(task):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds    = np.argmax(logits, axis=-1)
        f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
        acc      = accuracy_score(labels, preds)
        result   = {'f1_macro': f1_macro, 'accuracy': acc}
        if task == 'severity':
            result['mae'] = float(np.mean(np.abs(labels - preds)))
        return result
    return compute_metrics


def print_report(task, y_true, y_pred):
    if task == 'conflict':
        print(classification_report(y_true, y_pred,
              labels=[0,1], target_names=['no_conflict','conflict'], zero_division=0))
    elif task == 'severity':
        print(classification_report(y_true, y_pred,
              labels=[0,1,2], target_names=['mild','moderate','severe'], zero_division=0))
    else:
        print(classification_report(y_true, y_pred,
              labels=[0,1,2,3],
              target_names=['personal','political','sexual/gendered','threat'],
              zero_division=0))

In [12]:
# ── Train one task for one model ──────────────────────────────────────────────
def train_task(task, train_all, val_all, test_all, model_id, model_slug):
    from sklearn.metrics import f1_score as sk_f1
    label_col  = f'label_{task}'
    num_labels = 2 if task == 'conflict' else (3 if task == 'severity' else 4)
    batch_size = BATCH_SIZE_OVERRIDE.get(model_slug, BATCH_SIZE_DEFAULT)

    if task == 'conflict':
        train_rows, val_rows, test_rows = train_all, val_all, test_all
    else:
        train_rows = [r for r in train_all if r['label_conflict'] == 1]
        val_rows   = [r for r in val_all   if r['label_conflict'] == 1]
        test_rows  = [r for r in test_all  if r['label_conflict'] == 1]

    print(f'\n{"─"*60}')
    print(f'  Task: {task.upper()}  |  num_labels={num_labels}  |  '
          f'train:{len(train_rows)}  val:{len(val_rows)}  test:{len(test_rows)}')
    print(f'{"─"*60}')

    if len(train_rows) < 20:
        print('  [SKIP] Not enough samples.')
        return None, 0.0

    if task == 'type':
        train_rows = oversample_to_balance(train_rows, label_col, seed=SEED)
        oc = Counter(r[label_col] for r in train_rows)
        print(f'  After oversampling: {len(train_rows)} rows — ' +
              '  '.join(f'class {c}:{n}' for c, n in sorted(oc.items())))

    arr     = np.array([r[label_col] for r in train_rows])
    classes = np.unique(arr)
    weights = compute_class_weight('balanced', classes=classes, y=arr)
    class_weights = torch.tensor(weights, dtype=torch.float)
    print('  Class weights: ' + '  '.join(f'class {c}->{w:.3f}' for c, w in zip(classes, weights)))

    gamma = 0.0 if task == 'conflict' else (1.0 if task == 'severity' else 2.0)
    lr    = LR_BY_TASK[task]
    print(f'  Focal gamma: {gamma}  |  LR: {lr}  |  Batch: {batch_size}')

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    train_ds  = ThreadDataset(train_rows, tokenizer, MAX_LEN, label_col)
    val_ds    = ThreadDataset(val_rows,   tokenizer, MAX_LEN, label_col)
    test_ds   = ThreadDataset(test_rows,  tokenizer, MAX_LEN, label_col)

    model    = AutoModelForSequenceClassification.from_pretrained(
                   model_id, num_labels=num_labels, ignore_mismatched_sizes=True)
    task_out = Path(OUTPUT_DIR) / model_slug / task
    task_out.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir                  = str(task_out),
        num_train_epochs            = EPOCHS[task],
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = lr,
        warmup_ratio                = 0.15,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'no',
        load_best_model_at_end      = False,
        logging_steps               = 20,
        seed                        = SEED,
        report_to                   = 'none',
        fp16                        = torch.cuda.is_available(),
    )

    best_cb = BestModelInMemory()
    trainer = FocalLossTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = make_compute_metrics(task),
        callbacks       = [best_cb],
        class_weights   = class_weights,
        gamma           = gamma,
        task            = task,
    )

    print(f'  Training on {device}...')
    trainer.train()

    if best_cb.best_state is not None:
        model.load_state_dict({k: v.to(model.device) for k, v in best_cb.best_state.items()})
        print(f'  Loaded best model (f1={best_cb.best_f1:.4f}) from memory')
    else:
        print('  WARNING: best_state is None — using last epoch model')

    preds_out = trainer.predict(test_ds)
    y_pred    = np.argmax(preds_out.predictions, axis=-1).tolist()
    y_true    = [r[label_col] for r in test_rows]
    print(f'\n  Test set report ({task}):')
    print_report(task, y_true, y_pred)
    macro_f1 = sk_f1(y_true, y_pred, average='macro', zero_division=0)

    best_dir = task_out / 'best'
    best_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    print(f'  Saved -> {best_dir}')

    # Free GPU memory before next model
    del model, trainer
    torch.cuda.empty_cache()

    return best_dir, macro_f1

In [16]:
# ── Run all models × all tasks ────────────────────────────────────────────────
results = {}  # {model_slug: {task: macro_f1}}

for model_id, model_slug in MODELS_TO_COMPARE:
    print(f'\n{"="*60}')
    print(f'  MODEL: {model_id}  ({model_slug})')
    print(f'{"="*60}')
    results[model_slug] = {}
    for task in TASKS:
        try:
            _, f1 = train_task(task, train_all, val_all, test_all, model_id, model_slug)
            results[model_slug][task] = round(f1, 4)
        except Exception as e:
            print(f'  [ERROR] {task}: {e}')
            results[model_slug][task] = None

# ── Comparison table ──────────────────────────────────────────────────────────
print('\n\n' + '='*65)
print('  COMPARISON TABLE — Macro F1 on Test Set')
print('='*65)
print(f'  {"Model":<28} {"Conflict":>10} {"Severity":>10} {"Type":>10}')
print('  ' + '─'*58)
for model_id, model_slug in MODELS_TO_COMPARE:
    r = results.get(model_slug, {})
    def fmt(v): return f'{v:.3f}' if v is not None else '   N/A'
    print(f'  {model_id:<28} {fmt(r.get("conflict")):>10} {fmt(r.get("severity")):>10} {fmt(r.get("type")):>10}')
print('='*65)


  MODEL: xlm-roberta-base  (xlmr_base)

────────────────────────────────────────────────────────────
  Task: CONFLICT  |  num_labels=2  |  train:1168  val:249  test:252
────────────────────────────────────────────────────────────
  Class weights: class 0->0.750  class 1->1.501
  Focal gamma: 0.0  |  LR: 2e-05  |  Batch: 16


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training on GPU...


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,0.709799,0.637968,0.651688,0.674699
2,0.637713,0.645275,0.533527,0.534137
3,0.614841,0.621145,0.623567,0.626506
4,0.579346,0.585395,0.751368,0.783133
5,0.534300,0.582099,0.752976,0.783133


...

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,0.782703,0.782958,0.043981,0.072289
2,0.572819,0.423193,0.258741,0.746988
3,0.317205,0.479277,0.320554,0.469880
4,0.162459,0.348965,0.441937,0.686747
5,0.066141,0.457735,0.453181,0.795181
6,0.096991,0.796649,0.337080,0.554217
7,0.029342,0.563179,0.397074,0.746988
8,0.002208,0.630811,0.379622,0.722892
9,0.015301,0.681738,0.456446,0.783133
10,0.000134,0.789689,0.442100,0.771084


  * New best f1_macro: 0.0440 — saved in memory
  * New best f1_macro: 0.2587 — saved in memory
  * New best f1_macro: 0.3206 — saved in memory
  * New best f1_macro: 0.4419 — saved in memory
  * New best f1_macro: 0.4532 — saved in memory
  * New best f1_macro: 0.4564 — saved in memory
  Loaded best model (f1=0.4564) from memory



  Test set report (type):
                 precision    recall  f1-score   support

       personal       0.74      0.90      0.81        59
      political       0.43      0.43      0.43         7
sexual/gendered       0.00      0.00      0.00        16
         threat       0.00      0.00      0.00         2

       accuracy                           0.67        84
      macro avg       0.29      0.33      0.31        84
   weighted avg       0.55      0.67      0.60        84



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved -> /kaggle/working/models/xlmr_large/type/best


  COMPARISON TABLE — Macro F1 on Test Set
  Model                          Conflict   Severity       Type
  ──────────────────────────────────────────────────────────
  xlm-roberta-base                  0.780      0.389      0.344
  google/muril-base-cased           0.747      0.368      0.337
  xlm-roberta-large                 0.773      0.411      0.309


In [17]:
import shutil
from pathlib import Path

# Remove partial zip from failed attempt
for f in Path('/kaggle/working').glob('*.zip'):
  f.unlink()
  print(f'Removed: {f.name}')

# Zip xlmr_base and xlmr_large separately (muril_base already exists locally)
for model_slug in ['xlmr_base', 'xlmr_large']:
  zip_path = f'/kaggle/working/{model_slug}'
  shutil.make_archive(zip_path, 'zip', OUTPUT_DIR, model_slug)
  size_gb = Path(zip_path + '.zip').stat().st_size / 1e9
  _, used, free = shutil.disk_usage('/kaggle/working')
  print(f'{model_slug}.zip  {size_gb:.1f}GB  (free: {free/1e9:.1f}GB)')

Removed: xlmr_base.zip
Removed: xlmr_large.zip
xlmr_base.zip  2.5GB  (free: 5.4GB)
xlmr_large.zip  5.4GB  (free: 0.1GB)
